# Identify the watershed boundary shapefile

Run from `lab/`. Checks `SMF_stream.shp` and `SMF_voi.shp` in `init_data/shapefiles/` to determine
which (if either) is usable as the watershed boundary polygon for `model.update_solar_position()`.

Expectation per the tRIBS docs' definition of the `_voi` file type ("Mesh Voronoi Geometry — file
containing individual Voronoi polygon geometry"): `SMF_voi` should come back as Polygon geometry
with a large feature count (one per mesh node), and `SMF_stream` should come back as LineString
(the channel network, not a boundary) — which would rule it out immediately.

In [1]:
import geopandas as gpd

for name in ["SMF_stream", "SMF_voi"]:
    gdf = gpd.read_file(f"../init_data/shapefiles/{name}.shp")
    print(f"\n--- {name} ---")
    print("geometry type(s):", gdf.geom_type.unique())
    print("feature count:", len(gdf))
    print("CRS:", gdf.crs)
    print("total bounds:", gdf.total_bounds)


--- SMF_stream ---
geometry type(s): <StringArray>
['LineString', 'MultiLineString']
Length: 2, dtype: str
feature count: 149
CRS: EPSG:26912
total bounds: [ 394405.99996752 3686729.0000046   397802.49996752 3689521.5000046 ]

--- SMF_voi ---
geometry type(s): <StringArray>
['Polygon']
Length: 1, dtype: str
feature count: 3398
CRS: EPSG:26912
total bounds: [ 394372.73097  3686600.344681  397869.199694 3689751.793364]


## If `SMF_voi` confirms as Polygon geometry

It's individual mesh-cell polygons, not one boundary polygon — dissolve them into one shape.
Since the Voronoi cells tile the watershed with no gaps or overlaps, their union *is* the
watershed boundary.

Sanity check before trusting it: the printed centroid should land roughly in UTM 12N / EPSG 26912
easting 400,000–420,000, northing 3,690,000–3,700,000 (South Mountain Park, Phoenix, AZ, ballpark).
If it's wildly outside that range, stop — something isn't what we think it is.

In [2]:
voi = gpd.read_file("../init_data/shapefiles/SMF_voi.shp")
watershed_boundary = voi.union_all()   # geopandas >=0.14; use voi.unary_union on older versions

print("centroid:", watershed_boundary.centroid)
print("bounds:", watershed_boundary.bounds)

centroid: POINT (396358.3323027271 3688117.680830542)
bounds: (394372.73097, 3686600.344681, 397869.199694, 3689751.793364)
